In [ ]:
!pip install transformers[torch] datasets pandas scikit-learn


In [1]:
import pandas as pd
import re
import torch
import io
from google.colab import files  # Special tool for Colab uploads
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# ==========================================
# STEP 1: UPLOAD AND PREPARE DATASET
# ==========================================
print("Please upload your hospital_reviews.csv file:")
uploaded = files.upload() # This opens the upload button

# Load the file into a DataFrame
file_name = list(uploaded.keys())[0]
data = pd.read_csv(io.BytesIO(uploaded[file_name]))

# Handle Missing Data (from walkthrough)
data = data.dropna(subset=['text'])

# Cleaning/Normalization Function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data["cleaned_text"] = data["text"].apply(clean_text)

# Map labels to numbers
label_map = {"positive": 0, "neutral": 1, "negative": 2}
data["label"] = data["label"].map(label_map)

Please upload your hospital_reviews.csv file:


Saving hospital_reviews.csv to hospital_reviews.csv


In [3]:
# ==========================================
# STEP 2 & 3: TOKENIZE AND SPLIT
# ==========================================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Split: 80% Training, 20% Testing
train_df, test_df = train_test_split(data, test_size=0.2, random_state=42)

# Convert to Hugging Face format
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize_function(examples):
    return tokenizer(examples["cleaned_text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Remove non-numeric columns for the model
train_dataset = train_dataset.remove_columns([col for col in train_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']])
test_dataset = test_dataset.remove_columns([col for col in test_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [4]:
# ==========================================
# STEP 4 & 5: CONFIGURE & FINE-TUNE
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

# Step 4: Hyperparameters (The Rules)
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

# Step 5: The Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print("Starting Fine-tuning on your uploaded file...")
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting Fine-tuning on your uploaded file...


Epoch,Training Loss,Validation Loss
1,No log,1.010643
2,No log,1.018698
3,No log,1.022823


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3, training_loss=1.0872488021850586, metrics={'train_runtime': 34.2922, 'train_samples_per_second': 0.437, 'train_steps_per_second': 0.087, 'total_flos': 986675316480.0, 'train_loss': 1.0872488021850586, 'epoch': 3.0})

In [5]:
# ==========================================
# STEP 6: EVALUATE
# ==========================================
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)
labels = test_dataset["label"]

print(f"\nFinal Accuracy: {accuracy_score(labels, preds):.2f}")
print(f"Final F1 Score: {f1_score(labels, preds, average='weighted'):.2f}")


Final Accuracy: 1.00
Final F1 Score: 1.00


In [ ]:
# ==========================================
# STEP 7: SAVE
# ==========================================
model.save_pretrained("./fine_tuned_bert")
tokenizer.save_pretrained("./fine_tuned_bert")
print("\nSuccess! Your model trained on the CSV file is saved in the './fine_tuned_bert' folder.")